**optimization 1 :-**

    product_mes_ref_df is a pandas df, it first converted to spark df then again back 
    to pandas df. This is not necessary. We can directly use the pandas df to for insert_error_modular function 


In [0]:

  #spark.createDataFrame(product_mes_ref_df).createOrReplaceTempView("temp_data")
  #pdt = spark.table("temp_data").toPandas()
  pdt = product_mes_ref_df.reset_index(drop=True) 

**optimization 2 :-**

    if we broadcast mes data, then whole shuffle will not happen. Since PLC data has no duplicates then there  
    is no need of DINSTINCT. Also, prefixing columns for ambugity and readability.


In [0]:
  #join plc and mes data
  #spark.sql(f"""-- get sensors data for the product
  #create or replace temporary view plc_mes_udm_retrieve_unit as
  #SELECT DISTINCT
  #    -- PLC Columns
  #    TagId, Value, Timestamp, unix_timestamp, Date, Domain, Flag_ValueInRange,
  #    -- MES Columns
  #    ProductId, Machine, StartDateTimeMachineSpecific, EndDateTimeMachineSpecific, unix_StartDateTimeMachineSpecific, unix_EndDateTimeMachineSpecific,
  #    Length, Width, Thickness, Weight
  #    FROM temp_data src, plc_retrieve_coil plc WHERE TagId IS NOT NULL
  #    """) 
  


  spark.sql(f"""-- get sensors data for the product
  create or replace temporary view plc_mes_udm_retrieve_unit as
  SELECT /*+ BROADCAST(src) */
      -- PLC Columns
      plc.TagId, plc.Value, plc.Timestamp, plc.unix_timestamp, plc.Date, plc.Domain, plc.Flag_ValueInRange,
      -- MES Columns
      src.ProductId, src.Machine, src.StartDateTimeMachineSpecific, src.EndDateTimeMachineSpecific,
      src.unix_StartDateTimeMachineSpecific, src.unix_EndDateTimeMachineSpecific,
      src.Length, src.Width, src.Thickness, src.Weight
  FROM temp_data src
  CROSS JOIN plc_retrieve_coil plc
  WHERE plc.TagId IS NOT NULL
""")

**optimization 3 :-**

    unnecessary order by Timestamp at the start


In [0]:
df = spark.table("plc_retrieve_coil") 
  #.orderBy('Timestamp')

**optimization 4 :-**

    it will not run this two cast in the plan. Also, only required column are selected
    ( column puring) 

In [0]:

df = df.select(col("Timestamp").cast("timestamp").alias("Timestamp"), 
                 col("TagId"), col("Value").cast("float").alias("Value"))

**optimization 5 :-**

    Replace pivot with conditional aggregation (eliminates the pivot )
    conditional opeartion when() is faster than pivot(). 
    This uses first() with when() conditions instead of pivot, which is Photon-accelerated (databricks engine)
 
  

In [0]:
 pivot = df.groupBy("Timestamp") \
    .agg(
        first(when(col("TagId") == even_pass, col("Value")), ignorenulls=True).alias("evenPass"),
        first(when(col("TagId") == uneven_pass, col("Value")), ignorenulls=True).alias("unevenPass"),
        first(when(col("TagId") == pass_number, col("Value")), ignorenulls=True).alias("passNumber")
    ) \
    .orderBy('Timestamp') \
    .toPandas()

**optimization 6 :-**

    unnecessary pandas to spark to pandas conversion twice for positiondf
   

In [0]:
#positiondf = pd.concat(groups)
#dataframe = spark.createDataFrame( positiondf.reset_index())
#df = dataframe.toPandas()

  positiondf = pd.concat(groups)
  df = positiondf.reset_index()

**optimization 7 :-**

    No need for where clause cause plc_mes_udm_retrieve_unit only has required 
    productId. Also, no need to sort Timestamp because in next step we are doing a join which shuffles 
    the data anyway and no need to cast Timestamp to timestamp type. Also, no need to sort Timestamp  
                

  

In [0]:

  plc_mes_df = spark.sql('''
    SELECT TagId, Timestamp, Value, Date, unix_timestamp, Domain, ProductId, unix_StartDateTimeMachineSpecific, unix_EndDateTimeMachineSpecific, Flag_ValueInRange, Length
    FROM plc_mes_udm_retrieve_unit
    --WHERE productid in (select productId from temp_data);
  ''')
  #.withColumn("Timestamp", col("Timestamp").cast("timestamp")).sort("Timestamp")
  # print(plc_mes_df.count())

**optimization 8 :-**

    unnecessary cast and sorting Timestamp. Can 
    broadcast plc_mes_df because size around 5MB

    again unnecessary sorting of Timestamp. Can 
    broadcast data_master because of very small size



In [0]:
          
  first_merge_df = plc_mes_df \
    .join(sparkDF, ["Timestamp"], "right_outer") \
    #.withColumn("Timestamp",col("Timestamp").cast("timestamp")) \
    #.sort("Timestamp")
    
  data_master = spark.createDataFrame(sensor_relative_position_df)

  data_master = data_master.withColumnRenamed("Identifier","TagId")

  second_merge_df = broadcast(data_master)\
    .join(first_merge_df, ["TagId"], "right_outer")\
    #.sort("Timestamp")

**optimization 9 :-**

    optimization - repartition(1)  will only create 1 file per productId 


In [0]:
 
  #write result to a dir
  spark.table("result").repartition(1).write.format("parquet").mode("append").save(target_write_dir+f'/productId={ProductId}')